## About this notebook (2026 cleanup note)

**Original:** Team Tesla (Jayesh Suryavanshi, Sumeet Aher, Krati Sharma) — University at Buffalo NLP course project, 2022, run on Google Colab. Milestone 3, Task 3, stage 1 of 3: pulls unrest-related news via the Azure Bing News Search v7 API and extracts full article text/summary/keywords with `newspaper3k`, writing `getBingNews.csv` and `getBingNewsProcessed.csv`.

**This stage can no longer be re-run as written:** Microsoft retired the Bing Search APIs in August 2025. The code is kept as a historical record of the pipeline.

**Post-hoc edits made in the 2026 cleanup:**
- Removed a real (now inert) API subscription key that was committed in 2022; the code now reads `BING_SEARCH_V7_SUBSCRIPTION_KEY` from the environment.
- Fixed a loop-indentation bug: the article-writing loop sat *outside* the 500-page request loop, so only the final page of results was ever written to `getBingNews.csv`.
- Flagged the undefined `date` variable (it was set interactively in the original session).

All saved cell outputs are from the original 2022 runs and predate these fixes; nothing has been re-executed.

In [3]:
!pip install newspaper3k

import requests
import os
from IPython.display import HTML
import csv
import dateutil.parser
from newspaper import Article
import pandas

In [ ]:
heading=["datapublished","name","description","provider","url"]
with open('bingNews.csv', 'w') as csvFile:
    writer = csv.writer(csvFile)
    writer.writerow(heading)

In [ ]:
# NOTE (2026 cleanup): the original notebook hardcoded a real API key here; it has
# been removed. Microsoft retired the Bing Search APIs in August 2025, so this
# endpoint no longer exists -- the cell is kept as a historical record of the pipeline.
subscription_key = os.environ["BING_SEARCH_V7_SUBSCRIPTION_KEY"]
search_term = "protests+battles+explosions+violence+civilians+riots+violence"
search_url = "https://api.bing.microsoft.com/v7.0/news/search"

headers = {"Ocp-Apim-Subscription-Key" : subscription_key}

with open('getBingNews.csv', 'a') as csvFile:
    writer = csv.writer(csvFile)
    for offset in range(0,500):
      parameters  = {"q": search_term, "textDecorations": True, "textFormat": "HTML", 
      # NOTE (2026 cleanup): `date` (earliest publish date) was defined interactively
      # in the original Colab session and is undefined here; set it before running.
            "count": 100, "offset": offset, "mkt":"en-US", "since": date, "sortBy": "Date"}
      response = requests.get(search_url, headers=headers, params=parameters)
      response.raise_for_status()
      search_results = response.json()
      for article in search_results["value"]:
          array = [article["datePublished"],article["name"],article["description"],article["provider"],article["url"]]
          print(array)
          writer.writerow(array)

In [ ]:
df = pandas.read_csv('getBingNews.csv')
heading=["DatePublished","Name","Url","Text","Summary","Keywords"]
with open('getBingNewsProcessed.csv', 'w') as csvFile:
    writer = csv.writer(csvFile)
    writer.writerow(heading)
    
for index, row in df.iterrows():
    article = Article(row['url'])
    try:
        article.download()
        article.parse()
        article.nlp()
    except:
        continue
    date_pub = dateutil.parser.parse(row['datapublished'])
    with open('getBingNewsProcessed.csv', 'a') as csvFile:
        writer = csv.writer(csvFile)
        array = [date_pub.date().isoformat(),row['name'],row['url'], article.text,article.summary,article.keywords]
        writer.writerow(array)


/Users/dhayanidhigunasekaran/anaconda/lib/python3.5/site-packages/dateutil/parser/_parser.py:1204: UnknownTimezoneWarning: tzname IST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  category=UnknownTimezoneWarning)
